In [ ]:
ESTABLISH CONNECTION WITH POSTGRES SERVER AND IMPORT LIBRARIES

Source - Central Bank of Nigeria

In [ ]:
#i didn't install/uv add seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import requests
import psycopg2
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from sklearn.linear_model import LinearRegression



In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

In [ ]:
conn = psycopg2.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT
)
cur = conn.cursor()
print("Connected successfully")

Load dataframes

In [ ]:
df = pd.read_excel("NFEM_Rates_Data_in_Excel.xlsx")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.shape

Rename columns for postgres

In [ ]:
#turns all to lower case and removes spaces at the beginning and end
df.columns.str.strip().str.lower()
df = df.rename(columns={
    "ratedate": "date",
    "weightedAvgRate": "usd_ngn"
})

Create postgres tables

In [ ]:
cur.execute("""
CREATE TABLE IF NOT EXISTS fx_rates (
    date DATE,
    usd_ngn FLOAT
    );
    """)

In [ ]:
for i, row in df.iterrows():
    cur.execute("""
    INSERT INTO fx_rates (date, usd_ngn)
    VALUES (%s, %s)
    """, (row["date"], row["usd_ngn"]))

In [ ]:
conn.commit()